## Módulo 4. Configuración de categorías



In [ ]:
# ==========================================
# MODULO 4: CONFIGURACION DE CATEGORIAS
# ==========================================

# Definimos el archivo donde guardaremos las categorias

archivo_categorias = carpeta + "/categorias.csv"

# Creamos la funcion para configurar categorias

def configurar_categorias(usuario_activo):

    # Verificamos que exista una sesion activa

    if usuario_activo is None:

        print("Primero debes iniciar sesion.")
        return

    print("\nCONFIGURACION DE CATEGORIAS")
    print("---------------------------")

    # Creamos el archivo si todavia no existe

    if not os.path.exists(archivo_categorias):

        with open(archivo_categorias, "w", newline="", encoding="utf-8") as archivo:

            campos = ["id_usuario", "tipo", "categoria"]

            escritor = csv.DictWriter(archivo, fieldnames=campos)

            escritor.writeheader()

    # Mostramos el menu de opciones

    while True:

        print("\n1. Agregar categoria de ingreso")
        print("2. Agregar categoria de gasto")
        print("3. Ver mis categorias")
        print("4. Terminar")

        opcion = input("Selecciona una opcion: ")

        # Determinamos el tipo de categoria

        if opcion == "1":
            tipo = "Ingreso"

        elif opcion == "2":
            tipo = "Gasto"

        elif opcion == "3":

            with open(archivo_categorias, "r", encoding="utf-8") as archivo:

                lector = csv.DictReader(archivo)

                categorias = list(lector)

            print("\nMIS CATEGORIAS")

            for categoria in categorias:

                if categoria["id_usuario"] == str(usuario_activo["id_usuario"]):

                    print(categoria["tipo"], "-", categoria["categoria"])

            continue

        elif opcion == "4":

            print("Configuracion finalizada.")
            break

        else:

            print("Opcion no valida.")
            continue

        # Solicitamos el nombre de la categoria

        nombre_categoria = input("Nombre de la categoria: ").strip()

        if nombre_categoria == "":
            print("Debes escribir un nombre.")
            continue

        # Leemos las categorias existentes

        with open(archivo_categorias, "r", encoding="utf-8") as archivo:

            lector = csv.DictReader(archivo)

            categorias = list(lector)

        # Verificamos que no este repetida para ese usuario y tipo

        repetida = False

        for categoria in categorias:

            if (
                categoria["id_usuario"] == str(usuario_activo["id_usuario"])
                and categoria["tipo"] == tipo
                and categoria["categoria"].lower() == nombre_categoria.lower()
            ):

                repetida = True

        if repetida:

            print("Esta categoria ya existe.")
            continue

        # Guardamos la nueva categoria

        with open(archivo_categorias, "a", newline="", encoding="utf-8") as archivo:

            campos = ["id_usuario", "tipo", "categoria"]

            escritor = csv.DictWriter(archivo, fieldnames=campos)

            escritor.writerow({
                "id_usuario": usuario_activo["id_usuario"],
                "tipo": tipo,
                "categoria": nombre_categoria
            })

        print("Categoria guardada correctamente.")

## Módulo 5. Configuración del presupuesto


In [ ]:
# ==========================================
# MODULO 5: CONFIGURACION DEL PRESUPUESTO
# ==========================================

# Archivo donde guardaremos los presupuestos

archivo_presupuestos = carpeta + "/presupuestos.csv"


def configurar_presupuesto(usuario_activo):

    # Verificamos que exista una sesion activa

    if usuario_activo is None:
        print("Primero debes iniciar sesion.")
        return

    # Leemos las categorias de gasto del estudiante

    with open(archivo_categorias, "r", encoding="utf-8") as archivo:

        lector = csv.DictReader(archivo)

        categorias = [
            fila["categoria"]
            for fila in lector
            if fila["id_usuario"] == str(usuario_activo["id_usuario"])
            and fila["tipo"] == "Gasto"
        ]

    if not categorias:
        print("PRIMERO DEBES CREAR CATEGORIAS DE GASTOS.")
        return

    print("\nCONFIGURACION DEL PRESUPUESTO MENSUAL")
    print("-------------------------------------")

    # Solicitamos el presupuesto total

    while True:

        try:

            presupuesto_mensual = int(
                input("¿Cuanto dinero tienes disponible para el mes? $")
            )

            if presupuesto_mensual > 0:
                break

            print("El presupuesto debe ser mayor que cero.")

        except ValueError:

            print("Debes ingresar un numero entero.")

    # Distribuimos el presupuesto entre las categorias

    asignaciones = []
    total_asignado = 0

    print("\nDISTRIBUCION DEL PRESUPUESTO")

    for categoria in categorias:

        while True:

            try:

                valor = int(
                    input("Presupuesto para " + categoria + ": $")
                )

                if valor < 0:
                    print("No puedes ingresar valores negativos.")
                    continue

                if total_asignado + valor > presupuesto_mensual:

                    print("Superas el presupuesto mensual disponible.")
                    print("Te quedan: $", presupuesto_mensual - total_asignado)
                    continue

                break

            except ValueError:

                print("Debes ingresar un numero entero.")

        asignaciones.append({
            "categoria": categoria,
            "valor_categoria": valor
        })

        total_asignado += valor

    # Preparamos los registros del estudiante

    nuevos_registros = []

    for asignacion in asignaciones:

        nuevos_registros.append({
            "id_usuario": usuario_activo["id_usuario"],
            "presupuesto_mensual": presupuesto_mensual,
            "categoria": asignacion["categoria"],
            "valor_categoria": asignacion["valor_categoria"]
        })

    # Recuperamos los presupuestos de otros estudiantes

    registros_anteriores = []

    if os.path.exists(archivo_presupuestos):

        with open(archivo_presupuestos, "r", encoding="utf-8") as archivo:

            lector = csv.DictReader(archivo)

            for fila in lector:

                if fila["id_usuario"] != str(usuario_activo["id_usuario"]):

                    registros_anteriores.append(fila)

    # Actualizamos el archivo sin borrar los datos de otros usuarios

    campos = [
        "id_usuario",
        "presupuesto_mensual",
        "categoria",
        "valor_categoria"
    ]

    with open(archivo_presupuestos, "w", newline="", encoding="utf-8") as archivo:

        escritor = csv.DictWriter(archivo, fieldnames=campos)

        escritor.writeheader()

        escritor.writerows(registros_anteriores)

        escritor.writerows(nuevos_registros)

    # Mostramos el resultado

    print("\nPRESUPUESTO GUARDADO CORRECTAMENTE")
    print("---------------------------------")

    print("Presupuesto mensual: $", presupuesto_mensual)

    print("Total asignado: $", total_asignado)

    print("Dinero sin asignar: $", presupuesto_mensual - total_asignado)